In [1]:
import json, pandas as pd
from pathlib import Path

ROOT = Path().cwd().parent
labels = json.loads((ROOT / "configs/station_labels.json").read_text())

for sid, info in labels.items():
    df = pd.read_csv(ROOT / f"data/processed/client/{sid}.csv")
    
    # kWh divided by kW = a number between 0 and 1
    df["capacity_factor"] = df["total_produced_energy"] / info["capacity_kw"]
    
    cf = df["capacity_factor"]
    print(f'{info["label"]}  max={cf.max():.2f}  above_1={(cf > 1).sum()}  below_0={(cf < 0).sum()}')

S1  max=0.72  above_1=0  below_0=0
S2  max=0.80  above_1=0  below_0=0
S3  max=177664.58  above_1=1  below_0=0
S4  max=0.61  above_1=0  below_0=0
S5  max=0.66  above_1=0  below_0=0
S6  max=0.60  above_1=0  below_0=0
S7  max=3.70  above_1=86  below_0=0


## Finding the corrupted row in the s3 station

In [ ]:
by_label = {v["label"]: (k, v["capacity_kw"]) for k, v in labels.items()}

def load(label):
    sid, cap = by_label[label]
    df = pd.read_csv(ROOT / f"data/processed/client/{sid}.csv", parse_dates=["measured_ts"])
    df["capacity_factor"] = df["total_produced_energy"] / cap
    return df.sort_values("measured_ts").reset_index(drop=True)

s3 = load("S3")
i = s3.index[s3["capacity_factor"] > 1][0]
print(s3.loc[i-3:i+3, ["measured_ts", "total_produced_energy", "capacity_factor"]])

## Debugging the Bad rows in s7 station

In [4]:
s7 = load("S7")
bad = s7["capacity_factor"] > 1
gap = s7["measured_ts"].diff().dt.total_seconds() / 3600

print("hours since previous reading — BAD rows:")
print(gap[bad].describe().round(1))
print("\nhours since previous reading — NORMAL rows:")
print(gap[~bad].describe().round(1))

hours since previous reading — BAD rows:
count    86.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
Name: measured_ts, dtype: float64

hours since previous reading — NORMAL rows:
count    6712.0
mean        1.2
std         1.3
min         1.0
25%         1.0
50%         1.0
75%         1.0
max        77.0
Name: measured_ts, dtype: float64


## Rule for the capacity factor

In [6]:
def add_capacity_factor(df, capacity_kw):
    """Hourly kWh / rated kW. Above 1.0 is physically impossible -> blank it."""
    cf = df["total_produced_energy"] / capacity_kw
    df["capacity_factor"] = cf.where(cf <= 1.0)   # blank, don't delete
    return df

In [7]:
for label in [f"S{i}" for i in range(1, 8)]:
    sid, cap = by_label[label]
    df = pd.read_csv(ROOT / f"data/processed/client/{sid}.csv", parse_dates=["measured_ts"])
    add_capacity_factor(df, cap)
    cf = df["capacity_factor"]
    print(f"{label}  rows={len(df)}  max={cf.max():.2f}  blanked={cf.isna().sum()}")

S1  rows=2470  max=0.72  blanked=0
S2  rows=1627  max=0.80  blanked=0
S3  rows=1789  max=0.76  blanked=1
S4  rows=1769  max=0.61  blanked=0
S5  rows=1770  max=0.66  blanked=0
S6  rows=1759  max=0.60  blanked=0
S7  rows=6799  max=1.00  blanked=86
